2   序列模型

2.1理论计算题

![2.1理论计算题](2.1理论计算题.png){width=70%}

2.2编程题

In [18]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 步骤1：转小写 + 去除标点
    text_lower = text.lower()
    # 构建标点移除映射
    punc_table = str.maketrans('', '', string.punctuation)
    clean_text = text_lower.translate(punc_table)
    
    # 步骤2：按空格分词
    words = clean_text.split()
    if len(words) == 0:
        return {}, ([], [])
    
    # 步骤3：按频率构建词汇表（频率高ID更小，同频按出现先后）
    word_counter = Counter(words)
    # 排序：优先频率降序，频率相同保留首次出现顺序
    sorted_words = sorted(word_counter.keys(), key=lambda w: (-word_counter[w], words.index(w)))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 步骤4：滑动窗口生成特征与标签
    features = []
    labels = []
    max_start = len(words) - n
    for i in range(max_start + 1):
        window = words[i:i+n]
        features.append(window)
        # 取窗口后下一个词
        next_pos = i + n
        if next_pos < len(words):
            labels.append(words[next_pos])
        else:
            labels.append(None)
    
    return vocab, (features, labels)

# 测试示例（题目样例输入）
if __name__ == "__main__":
    input_text = "The time machine"
    vocab, (feat, lab) = preprocess_text(input_text, n=2)
    print("词汇表：", vocab)
    print("特征列表：", feat)
    print("标签列表：", lab)

词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征列表： [['the', 'time'], ['time', 'machine']]
标签列表： ['machine', None]


3 循环神经网络

3.1理论计算题

![3.1理论计算题](3.1理论计算题.jpg){width=70%}

3.2编程题

In [19]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播
    :param x_t: (batch_size, input_size)
    :param h_prev: (batch_size, hidden_size)
    :param W_hx: (hidden_size, input_size)
    :param W_hh: (hidden_size, hidden_size)
    :param b_h: (1, hidden_size)
    :return h_t: 当前隐状态 (batch_size, hidden_size)
    :return cache: 反向传播缓存 (x_t, h_prev, z_t, h_t)
    """
    z_t = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    h_t = np.tanh(z_t)
    cache = (x_t, h_prev, z_t, h_t)
    return h_t, cache


def rnn_backward(dh_next, cache, W_hx, W_hh):
    """
    RNN单步反向传播，计算全部梯度
    :param dh_next: 损失对h_t的上游梯度 (batch_size, hidden_size)
    :param cache: 前向缓存 (x_t, h_prev, z_t, h_t)
    :param W_hx: (hidden_size, input_size)
    :param W_hh: (hidden_size, hidden_size)
    :return dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t, h_prev, z_t, h_t = cache
    batch_size = x_t.shape[0]

    # tanh梯度
    dz_t = dh_next * (1 - np.square(h_t))

    # 各变量梯度
    dx_t = dz_t @ W_hx
    dh_prev = dz_t @ W_hh
    dW_hx = dz_t.T @ x_t
    dW_hh = dz_t.T @ h_prev
    db_h = np.sum(dz_t, axis=0, keepdims=True)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ------------------- 测试示例 -------------------
if __name__ == "__main__":
    # 超参
    batch_size = 2
    input_size = 3
    hidden_size = 4

    # 随机初始化输入、隐状态、权重
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(hidden_size, input_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(1, hidden_size)

    # 前向传播
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("h_t shape:", h_t.shape)

    # 模拟上游梯度（损失对h_t的导数）
    dh_next = np.random.randn(batch_size, hidden_size)
    # 反向传播求梯度
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache, W_hx, W_hh)

    # 打印各梯度维度校验
    print("dx_t shape:", dx_t.shape)       # (2,3)
    print("dh_prev shape:", dh_prev.shape) # (2,4)
    print("dW_hx shape:", dW_hx.shape)     # (4,3)
    print("dW_hh shape:", dW_hh.shape)     # (4,4)
    print("db_h shape:", db_h.shape)       # (1,4)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (1, 4)


4 高级循环神经网络

4.1理论计算题

![4.1理论计算题](4.1理论计算题.png)

4.2编程题

In [20]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # 双向单层RNN，默认输入shape (seq_len, batch, input_size)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            num_layers=1
        )

    def forward(self, X):
        """
        参数：
            X: 输入序列，shape (seq_len, batch_size, input_dim)
        返回：
            seq_hidden: 每个时间步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            global_hidden: 全局序列表示 (batch, 2*hidden_dim)
        """
        # out: (seq_len, batch, 2*hidden_dim)  每一步拼接前向+后向隐状态
        # hn: (num_layers*2, batch, hidden_dim)
        out, hn = self.rnn(X)
        
        # 拆分前向、后向最后时刻隐状态
        h_forward = hn[0]   # 前向最后一步 (batch, hidden_dim)
        h_backward = hn[1]  # 后向最后一步 (batch, hidden_dim)
        # 拼接得到全局序列表示
        global_hidden = torch.cat([h_forward, h_backward], dim=-1)
        
        return out, global_hidden


# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    # 超参
    seq_len = 10
    batch_size = 4
    input_dim = 8
    hidden_dim = 16
    
    # 构造输入 (seq_len, batch, input_dim)
    X = torch.randn(seq_len, batch_size, input_dim)
    # 初始化编码器
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    # 前向传播
    seq_h, global_h = encoder(X)
    
    print("时序拼接隐状态 shape:", seq_h.shape)    # torch.Size([10, 4, 32])
    print("全局序列表示 shape:", global_h.shape)  # torch.Size([4, 32])

时序拼接隐状态 shape: torch.Size([10, 4, 32])
全局序列表示 shape: torch.Size([4, 32])


5 嵌入向量

![5.1理论计算题](5.1理论计算题.png){width=70%}

5.2编程题

In [21]:
import numpy as np

def cbow_forward_loss(context_batch, target_batch, W, W_out):
    """
    CBOW 前向传播 + 完整Softmax交叉熵损失
    参数：
        context_batch: (batch_size, context_size) 上下文词索引
        target_batch: (batch_size,) 中心词索引标签
        W: (V, d) 输入嵌入矩阵
        W_out: (d, V) 输出投影权重
    返回：
        loss: 标量，批次平均交叉熵损失
    """
    batch_size, context_size = context_batch.shape
    V, d = W.shape

    # 1. 获取上下文词嵌入 (B, C, d)
    context_embeds = W[context_batch]
    # 2. 上下文向量求平均得到隐层 h (B, d)
    h = np.sum(context_embeds, axis=1) / context_size

    # 3. 计算词汇得分 (B, V)
    score = h @ W_out

    # 4. 数值稳定完整Softmax
    row_max = np.max(score, axis=1, keepdims=True)
    exp_score = np.exp(score - row_max)
    softmax_prob = exp_score / np.sum(exp_score, axis=1, keepdims=True)

    # 5. 取出每个样本目标词预测概率
    batch_idx = np.arange(batch_size)
    target_p = softmax_prob[batch_idx, target_batch]

    # 6. 交叉熵损失，批次求平均
    log_p = -np.log(target_p + 1e-10)
    loss = np.sum(log_p) / batch_size

    return loss

# ----------------测试示例----------------
if __name__ == "__main__":
    # 超参设置
    vocab_size = 12    # V
    embed_dim = 3      # d
    batch = 4
    c_size = 2         # context_size

    # 模拟输入数据
    ctx = np.array([[0, 3], [1, 5], [2, 7], [4, 9]])
    tgt = np.array([2, 4, 6, 8])

    # 随机初始化权重
    W = np.random.randn(vocab_size, embed_dim)
    W_out = np.random.randn(embed_dim, vocab_size)

    loss_val = cbow_forward_loss(ctx, tgt, W, W_out)
    print(f"CBOW批次平均损失：{loss_val:.4f}")

CBOW批次平均损失：4.0814


6 注意力机制

6.1理论计算题

![6.1理论计算题](6.1理论计算题.png)

6.2编程题

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # 题目固定超参
        self.num_heads = 2
        self.d_model = 4
        self.d_k = self.d_model // self.num_heads  # d_k=2

        # Q/K/V 投影层：输入d_model，输出d_model
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        # 多头拼接后最终线性层
        self.w_o = nn.Linear(self.d_model, self.d_model)

    def scaled_dot_product_attn(self, q, k, v):
        """
        单头缩放点积注意力
        q/k/v shape: (seq_len, batch, d_k)
        """
        attn_score = torch.matmul(q, k.transpose(-1, -2)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        out = torch.matmul(attn_weight, v)
        return out

    def split_heads(self, x):
        """
        x: (seq_len, batch, d_model)
        return: (num_heads, seq_len, batch, d_k)
        """
        seq_len, batch, _ = x.shape
        x = x.view(seq_len, batch, self.num_heads, self.d_k)
        # 调换维度到头部在前
        return x.permute(2, 0, 1, 3)

    def concat_heads(self, x):
        """
        x: (num_heads, seq_len, batch, d_k)
        return: (seq_len, batch, d_model)
        """
        num_h, seq_len, batch, _ = x.shape
        x = x.permute(1, 2, 0, 3)
        return x.contiguous().view(seq_len, batch, self.d_model)

    def forward(self, X):
        """
        输入X shape: (seq_len, batch, d_model)
        返回输出 shape: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # 1. 线性投影Q K V
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)

        # 2. 拆分为多头
        Q_h = self.split_heads(Q)  # (2, seq_len, batch, 2)
        K_h = self.split_heads(K)
        V_h = self.split_heads(V)

        # 3. 每个头单独计算注意力
        attn_out = self.scaled_dot_product_attn(Q_h, K_h, V_h)  # (2, seq_len, batch, 2)

        # 4. 拼接所有头
        concat_out = self.concat_heads(attn_out)  # (seq_len, batch, 4)

        # 5. 最终线性变换
        output = self.w_o(concat_out)
        return output


# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    # 模拟输入
    seq_len = 5
    batch_size = 3
    d_model = 4
    X = torch.randn(seq_len, batch_size, d_model)

    mha = MultiHeadAttention()
    out = mha(X)

    print("输入形状:", X.shape)    # torch.Size([5, 3, 4])
    print("输出形状:", out.shape)  # torch.Size([5, 3, 4])

输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
